
<a href="https://colab.research.google.com/github/kokchun/Databehandling-AI22/blob/main/Lectures/L1-missing-aggregation.ipynb" target="_parent"><img align="left" src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> &nbsp; for interacting with the code

---
# Lecture notes - Pandas - missing data & aggregations

---
This is the lecture note for **Pandas missing data & aggregations** - but it's built upon contents from previous course: 
- Python programming

<p class = "alert alert-info" role="alert"><b>Note</b> that this lecture note gives a brief introduction to Pandas. I encourage you to read further about pandas.

Read more 

- [documentation - read_csv](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html)

- [documentation - aggregate](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.aggregate.html)

- [documentation - groupby](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html)

- [documentation - missing values](https://pandas.pydata.org/pandas-docs/stable/user_guide/missing_data.html)
---

## Setup
- create dummy DataFrame
- it's very useful to create dummy DataFrame in order to test and learn Pandas methods

In [ ]:
import numpy as np 
import pandas as pd
np.random.seed(42)

n = 8
random_matrix = np.random.randint(1,10, (n,n))
index = np.random.choice(random_matrix.size, 10, replace = False)
print(f"index {index}")
print(f"Flattens the matrix: {random_matrix.ravel()}")

random_matrix = random_matrix*1.0 # to convert it to float as np.nan is float
random_matrix.ravel()[index] = None # returns contiguous flattened array
# numeric containers always use NaN 
print(random_matrix)

In [ ]:
scores = pd.DataFrame(random_matrix, index=[
                      f"Player {i}" for i in range(1, n+1)], columns=[f"Round {i}" for i in range(1,n+1)])
scores

---
## Missing data techniques
- None - Pythonic missing data. A Python object - operations done on Python level and not numpy or pandas.
    - Error when performing arithmetic on None dtypes
- NaN - floating point "Not a Number". Supports fast operations, however all arithmetic operations with NaN results in a NaN
- NA - pd.NA -> missing indicator that can be used consistently across data types

Pandas treats NaN and None almost interchangeably
- isnull() - returns true if value is null  
- notnull() - returns true if value is not null
- dropna() - drops the the whole axis with nulls 
- fillna() - fills the null values with certain value

In [ ]:
scores.isnull() 

In [ ]:
scores.dropna() # removes all rows that contains any NaNs

In [ ]:
scores.dropna(axis="columns") # removes all columns that contains any NaNs

In [ ]:
scores.fillna(0) # fills all nulls with zero 

---
## Missing data - strategy
Note that there are several strategies to deal with missing data and we will come back to this in the future. 
The strategy you should choose may depend on:
- dataset size
    - small dataset - must be more careful with the data
    - large dataset and small amounts of nulls - can probably remove the rows
- can use different ways to impute missing data
- valuable information (depends on the usage of the data)
- percentage of missing values 
- domain knowledge - how to fill the nulls 
    - e.g. the score example above might be plausible to assume 0 score if the value is missing for certain round
    - e.g. missing some value in a category might be filled with mean or median of that category
    - e.g. missing value between two points could be interpolated
- note that missing values can negatively impact:
    - data visualization
    - arithmetic computations
    - machine learning algorithms 

In [ ]:
import seaborn as sns 
# Seaborn has some toy datasets for learning: https://github.com/mwaskom/seaborn-data
titanic = sns.load_dataset("titanic")
titanic.head()

In [ ]:
nulls = titanic.isnull().sum()
print(nulls) 

In [ ]:
# Histogram of the age distribution for male and female 
sns.histplot(data= titanic, x="age", kde=True, hue="sex")

In [ ]:
# Imagine that we have talked to a historian and this person advices us to use median age of the corresponding gender to fill in the missing ages 

median_male_age = titanic[titanic["sex"] == "male"]["age"].median()
median_female_age = titanic[titanic["sex"] == "female"]["age"].median()

print(f"Median male age: {median_male_age}")
print(f"Median female age: {median_female_age}")


titanic.loc[titanic["sex"] == "male", "age"] = titanic.loc[titanic["sex"] == "male", "age"].fillna(median_male_age)
titanic.loc[titanic["age"]==29, "sex"].value_counts() # to see that we get desired results

In [ ]:
print(titanic.isnull().sum().loc["age"]) # now we have 53 nulls left for age 
titanic.loc[titanic["age"].isnull(), "sex"].value_counts() # all these are female 

In [ ]:
titanic["age"] = titanic["age"].fillna(median_female_age)
titanic.isnull().sum().sex

In [ ]:
# now we see a huge peak at around 30 years, which may not reflect the reality very well 
# Can you figure out other ways of filling in the missing data 
sns.histplot(data=titanic, x="age", kde=True, hue="sex")

---
## Aggregations and groupby
Summarizing data

aggegations: 
```python
sum(), mean(), median(), min(), max()
```

Don't confuse with the **aggregate()** or **agg()** for short, method which uses one or more operations over specified axis

In [ ]:
survivors = titanic.groupby("sex")[["survived", "age"]].mean()
survivors
for i in survivors.index:
    survived_percentage = survivors.loc[i, 'survived']*100
    print(f"Percentage survived {i}: {survived_percentage:.1f}%")
    print(f"Mean age of survived {i}: {survivors.loc[i, 'age']:.1f} years")

In [ ]:
# important to select columns that are valid for max() - will raise TypeError in the future otherwise
titanic.groupby("class", observed=True)[["age", "fare"]].max() 

In [ ]:
# aggregation with different aggregation methods 
titanic.select_dtypes(include="number").agg(["count","mean","median","min","max"])

In [ ]:
# survived and class 
survived_class = titanic.groupby(["survived", "pclass"]).count() # many from 3rd class died, counts non-null values
survived_class = survived_class.reset_index()
survived_class
ax = sns.barplot(data = survived_class, x="pclass", y ="sex", hue="survived")
ax.set(ylabel="count", title="Survival count in titanic vs class")

---

Kokchun Giang

[LinkedIn][linkedIn_kokchun]

[GitHub portfolio][github_portfolio]

[linkedIn_kokchun]: https://www.linkedin.com/in/kokchungiang/
[github_portfolio]: https://github.com/kokchun/Portfolio-Kokchun-Giang

---